<a href="https://colab.research.google.com/github/dennisgathu8/36CHAMBERS/blob/main/AI_AGENT(Agentic_AI).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade yfinance
!pip install pandas
!pip install numpy
!pip install torch

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque

In [ ]:
symbol = "AAPL"
start_date = "2021-01-01"
end_date = "2025-02-18"
data = yf.download(symbol, start=start_date, end=end_date)
print(data.head())
print(data.index)
print(data.empty)

[*********************100%***********************]  1 of 1 completed

Price            Close        High         Low        Open     Volume
Ticker            AAPL        AAPL        AAPL        AAPL       AAPL
Date                                                                 
2021-01-04  126.405235  130.507713  123.816764  130.419806  143301900
2021-01-05  127.968109  128.681170  125.448012  125.897338   97664900
2021-01-06  123.660469  128.007149  123.445576  124.754466  155088000
2021-01-07  127.880150  128.573671  124.891203  125.379593  109578200
2021-01-08  128.983932  129.550467  127.206184  129.355099  105158200
DatetimeIndex(['2021-01-04', '2021-01-05', '2021-01-06', '2021-01-07',
               '2021-01-08', '2021-01-11', '2021-01-12', '2021-01-13',
               '2021-01-14', '2021-01-15',
               ...
               '2025-02-03', '2025-02-04', '2025-02-05', '2025-02-06',
               '2025-02-07', '2025-02-10', '2025-02-11', '2025-02-12',
               '2025-02-13', '2025-02-14'],
              dtype='datetime64[ns]', name='Date',

In [ ]:
# feature engineering
data['SMA_5'] = data['Close'].rolling(window=5).mean()
data['SMA_20'] = data['Close'].rolling(window=20).mean()
data['Returns'] = data['Close'].pct_change()

In [ ]:
data.dropna(inplace=True)
data.reset_index(drop=True, inplace=True)

In [ ]:
# define action space
ACTIONS = {0: "HOLD", 1: "BUY", 2: "SELL"}
# get state  function
def get_state(data, index):
  return np.array([
      float(data.loc[index, 'Close']),
      float(data.loc[index, 'SMA_5']),
      float(data.loc[index, 'SMA_20']),
      float(data.loc[index, 'Returns'])
  ])


In [22]:
class TradingEnvironment:
    def __init__(self, data):
        self.data = data
        self.initial_balance = 10000
        self.balance = self.initial_balance
        self.holdings = 0
        self.index = 0

    def reset(self):
        self.balance = self.initial_balance
        self.holdings = 0
        self.index = 0
        return get_state(self.data, self.index)

    def step(self, action):
        price = self.data.iloc[self.index]['Close'].item()  # Fixed line
        reward = 0

        if action == 1 and self.balance >= price:  # BUY
            self.holdings = self.balance // price
            self.balance -= self.holdings * price
        elif action == 2 and self.holdings > 0:  # SELL
            self.balance += self.holdings * price
            self.holdings = 0

        self.index += 1
        done = self.index >= len(self.data) - 1

        if done:
            reward = self.balance - self.initial_balance

        next_state = get_state(self.data, self.index) if not done else np.zeros(4)
        return next_state, reward, done, {}

def get_state(data, index):
    return data.iloc[index][['Open', 'High', 'Low', 'Close']].values

In [23]:
# Deep Q-Network
class DQN(nn.Module):
    def __init__(self, state_size, action_size):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(state_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_size)

    def forward(self, x):  # Fixed typo and layer order
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

In [24]:
# DQN Agent
class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=2000)
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.001
        self.epsilon_decay = 0.995
        self.learning_rate = 0.001
        self.model = DQN(state_size, action_size)
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        self.criterion = nn.MSELoss()  # Added loss function

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def act(self, state):
        if random.uniform(0, 1) < self.epsilon:
            return random.randrange(self.action_size)  # Fixed ACTIONS issue
        state = torch.FloatTensor(state).unsqueeze(0)
        with torch.no_grad():
            q_values = self.model(state)
        return torch.argmax(q_values).item()

    def replay(self, batch_size):
        if len(self.memory) < batch_size:
            return
        minibatch = random.sample(self.memory, batch_size)

        for state, action, reward, next_state, done in minibatch:
            target = reward
            if not done:
                next_state_tensor = torch.FloatTensor(next_state).unsqueeze(0)  # Fixed typo
                target += self.gamma * torch.max(self.model(next_state_tensor)).item()

            state_tensor = torch.FloatTensor(state).unsqueeze(0)
            target_tensor = self.model(state_tensor).clone().detach()  # Fixed typo
            target_tensor[0][action] = target

            self.optimizer.zero_grad()
            output = self.model(state_tensor)
            loss = self.criterion(output, target_tensor)  # Fixed undefined criterion
            loss.backward()
            self.optimizer.step()

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

In [25]:
# Train the agent
env = TradingEnvironment(data)
agent = DQNAgent(state_size=4, action_size=3)
batch_size = 32
episodes = 500
total_rewards = []

for episode in range(episodes):
    state = env.reset()
    done = False
    total_reward = 0

    while not done:
        action = agent.act(state)
        next_state, reward, done, _ = env.step(action)
        agent.remember(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward

    agent.replay(batch_size)
    total_rewards.append(total_reward)
    print(f"Episode {episode+1}/{episodes}, Total Reward: {total_reward}")

print("Training Complete!")

Episode 1/500, Total Reward: -9857.377754211426
Episode 2/500, Total Reward: 2542.3767166137695
Episode 3/500, Total Reward: -9859.8560256958
Episode 4/500, Total Reward: -9822.4174118042
Episode 5/500, Total Reward: -9840.596710205078
Episode 6/500, Total Reward: -9879.221351623535
Episode 7/500, Total Reward: -9961.557632446289
Episode 8/500, Total Reward: -470.9794235229492
Episode 9/500, Total Reward: -9967.990737915039
Episode 10/500, Total Reward: -9799.07054901123
Episode 11/500, Total Reward: 12920.812210083008
Episode 12/500, Total Reward: -9978.197731018066
Episode 13/500, Total Reward: -9992.95662689209
Episode 14/500, Total Reward: -9795.028106689453
Episode 15/500, Total Reward: -9806.954208374023
Episode 16/500, Total Reward: -9820.833450317383
Episode 17/500, Total Reward: -9763.713996887207
Episode 18/500, Total Reward: -9850.706604003906
Episode 19/500, Total Reward: 4538.985023498535
Episode 20/500, Total Reward: -9824.739921569824
Episode 21/500, Total Reward: -9869.

In [26]:
# create a fresh environment instance for testing
test_env = TradingEnvironment(data)
state = test_env.reset()
done = False

# simulate a trading session using the trained Agent
while not done:
  action = agent.act(state)
  next_state, reward, done, _ = test_env.step(action)
  state = next_state if next_state is not None else state

final_balance = test_env.balance
profit = final_balance - test_env.initial_balance
print(f"Final Balance after testing: ${final_balance:.2f}")
print(f"Total Profit: ${profit:.2f}")

Final Balance after testing: $59.53
Total Profit: $-9940.47
